# Film collection: a five-decade coverage check

Provisional window: **1950 <= page year < 2000**. This includes the 1950s through the 1990s, not the year 2000.

This first run checks five candidate measurement pages per decade. It does not choose the final analytical sample or test a trend. The directory is ordered by submission date; its first page is a convenience frame, not a representative sample.

Country names and years below are source claims, not independently verified production countries or original release years. Short films, excerpts and repeat measurements may be present. All candidates remain pending.

The directory counts measurements, not distinct eligible films. Its public client script exposes a read-only OCHRE query. This notebook uses that query structure, adds an integer page-year condition, and keeps the site's 50-row first-page limit. The request body is retained in the log.

Start Jupyter in the project root. Both `collect` and `review_live` default to `False` to replay saved responses. The separate Titanic notebook is a short demonstration.

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import requests
from bs4 import BeautifulSoup
import json
import re
import time
import hashlib

folder = Path.cwd()
if not (folder / "01_raw").is_dir():
    raise RuntimeError("Start Jupyter from the project root, beside the 01_raw folder.")
collect = False
decades = [1950, 1960, 1970, 1980, 1990]
probe_size = 5  # Request budget for a diagnostic batch, not a research quota.
base = folder / "01_raw/coverage_1950_1999"
latest = base / "latest_run.txt"
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") if collect else latest.read_text().strip()
run = base / run_id
run.mkdir(parents=True, exist_ok=True)
session = requests.Session()
session.headers["User-Agent"] = "ADS206-CW1-coverage-check/0.1 (educational; limited requests)"
print("Mode:", "live collection" if collect else "saved responses")

print("Evidence run:", run_id)


Mode: saved responses
Evidence run: 20261107T164214300775Z


## Requests and evidence

Save each received response and log every attempt. Retry temporary network errors, HTTP 429 and 5xx only. Other HTTP failures stop collection. The pause, timeout and retry limit are operational settings, not analytical parameters.

The linked OCHRE terms were reviewed before this limited run. Robots pages currently return 404; this is not explicit scraping permission. Policy responses and the public directory client are saved below.

In [2]:
def fetch(url, filename, query=None, allow_missing=False, cache=None, online=None, interval=1.2):
    store = run if cache is None else cache
    active = collect if online is None else online
    if not active:
        return (store / filename).read_bytes()
    for attempt in range(3):
        started = datetime.now(timezone.utc).isoformat()
        clock = time.perf_counter()
        tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
        saved_name = f"{Path(filename).stem}_{tag}_attempt_{attempt + 1}{Path(filename).suffix}"
        entry = {"time_utc": started, "url": url, "retry_count": attempt,
                 "method": "POST" if query else "GET",
                 "file": saved_name, "cache_file": filename, "collector": "AI-assisted coverage check"}
        if query is not None:
            entry["request_body"] = query
        response = None
        try:
            response = session.request("POST" if query else "GET", url,
                data=query.encode("utf-8") if query else None,
                headers={"Content-Type": "application/xquery"} if query else None,
                timeout=(10, 45))
            (store / saved_name).write_bytes(response.content)
            entry.update(status=response.status_code,
                sha256=hashlib.sha256(response.content).hexdigest(),
                final_url=response.url,
                content_type=response.headers.get("Content-Type"),
                retry_after=response.headers.get("Retry-After"),
                redirects=[r.status_code for r in response.history])
        except requests.RequestException as error:
            entry["error"] = type(error).__name__
        finally:
            entry["seconds"] = round(time.perf_counter() - clock, 3)
            with (store / "requests.jsonl").open("a", encoding="utf-8") as file:
                file.write(json.dumps(entry) + "\n")
            time.sleep(interval)
        if response is not None and (response.status_code < 400 or
                                     allow_missing and response.status_code == 404):
            if saved_name != filename:
                (store / filename).write_bytes(response.content)
            return response.content
        temporary = response is None or response.status_code == 429 or response.status_code >= 500
        if not temporary or attempt == 2:
            if response is not None:
                response.raise_for_status()
            raise ConnectionError("Request failed; see requests.jsonl.")
        delay = 2 ** attempt
        if response is not None and response.status_code == 429:
            retry_after = response.headers.get("Retry-After", "")
            if not retry_after.isdigit():
                raise RuntimeError("Rate limited: review Retry-After before continuing.")
            delay = max(delay, int(retry_after))
            if delay > 60:
                raise RuntimeError("Rate limited: stop and resume later.")
        time.sleep(delay)


In [3]:
from urllib.robotparser import RobotFileParser

for host, target in [
    ("https://cinemetrics.uchicago.edu", "/movie/"),
    ("https://ochre.lib.uchicago.edu", "/ochre/v2/ochre.php")
]:
    name = "cinemetrics" if "cinemetrics" in host else "ochre"
    body = fetch(host + "/robots.txt", name + "_robots.txt", allow_missing=True)
    if body.lstrip().lower().startswith((b"<!doctype", b"<html")):
        print(name, "robots page unavailable; no permission is inferred.")
    else:
        rules = RobotFileParser()
        rules.parse(body.decode("utf-8", errors="replace").splitlines())
        if not rules.can_fetch(session.headers["User-Agent"], host + target):
            raise RuntimeError("robots.txt disallows this path. Stop collection.")
fetch("https://ochre.lib.uchicago.edu/ods/termsOfUse.html", "terms_of_use.html")
fetch("https://cinemetrics.uchicago.edu/_next/static/chunks/08y3_m-grp-_d.js", "directory_client.js")
print("Policy pages and public directory client saved.")


cinemetrics robots page unavailable; no permission is inferred.
ochre robots page unavailable; no permission is inferred.
Policy pages and public directory client saved.


## Discover candidate URLs

The property UUIDs and collection scope come from the Cinemetrics directory client script:
https://cinemetrics.uchicago.edu/_next/static/chunks/08y3_m-grp-_d.js

Only the first page is requested for each decade. Counts refer to exact integer years in the directory, not search-text hits. Selection uses title/year/director text to avoid exact repeated labels within this probe; this is not verified film-level deduplication. No ASL/MSL values are used to choose candidates.

In [4]:
path = "interpretations/interpretation/properties/property/property"
def field(uid):
    return f'string-join($film/{path}[label/@uuid = "{uid}"]/value/(content[@xml:lang = "eng"]/string, .[empty(content)]), " ")'
fields = {
    "title": 'string-join($film/identification/label/content[@xml:lang = "eng"]/string, " ")',
    "year": field("ceaecbc7-d1bc-4a6a-8698-7193f07c4f2a"),
    "director": field("6ce91c0c-34c5-46e4-9e22-434269f089d4"),
    "country": field("900560b7-7279-4f4d-bf0f-3393a346ed76")}
scope = 'cts:and-query((cts:element-attribute-value-query(xs:QName("ochre"), xs:QName("uuidBelongsTo"), "30a6cec8-849f-4c03-be3a-469cdb749398"), cts:element-attribute-value-query(xs:QName("tree"), xs:QName("uuid"), "5f69ef7c-9006-4e7a-bf56-0f266346f9fb")))'
filters = f'[parent::ochre/@uuidBelongsTo = "30a6cec8-849f-4c03-be3a-469cdb749398"][context/context/tree/@uuid = "5f69ef7c-9006-4e7a-bf56-0f266346f9fb"][{path}]'
endpoint = 'https://ochre.lib.uchicago.edu/ochre/v2/ochre.php?xquery&xsl=none&format=json&lang="*"'

def catalog_query(start):
    return "\n".join([
        'xquery version "1.0-ml";',
        f'let $films := cts:search(fn:collection("ochre/concept")/ochre/concept, {scope}, ("unfiltered", cts:index-order(cts:element-attribute-reference(xs:QName("interpretation"), xs:QName("date")), "descending"))){filters}',
        f'let $matching := for $film in $films let $year := normalize-space({fields["year"]}) where (if ($year castable as xs:integer) then (xs:integer($year) ge {start} and xs:integer($year) lt {start+10}) else false()) return $film',
        'let $rows := for $film in subsequence($matching, 1, 50) return <film uuid="{$film/@uuid}">' + ''.join(f'<{key}>{{{value}}}</{key}>' for key, value in fields.items()) + '</film>',
        'return <ochre><rows totalCount="{count($matching)}">{$rows}</rows></ochre>'])

manifest = []
coverage = []
for decade in decades:
    raw = fetch(endpoint, f"catalog_{decade}.json", catalog_query(decade))
    rows = json.loads(raw)["result"]["ochre"]["rows"]
    candidates = rows.get("film", [])
    if isinstance(candidates, dict):
        candidates = [candidates]
    coverage.append({"decade": decade, "directory_measurements": rows.get("totalCount"),
                     "first_page_rows": len(candidates)})
    seen = set()
    for position, row in enumerate(candidates, 1):
        title = row.get("title")
        if not isinstance(title, str) or not title.strip():
            continue
        key = (title.strip().casefold(), row.get("year"), str(row.get("director", "")).casefold())
        if key in seen:
            continue
        seen.add(key)
        manifest.append({"decade": decade, "directory_row": position, **row,
                         "url": "https://cinemetrics.uchicago.edu/movie/" + row["uuid"]})
        if len(seen) == probe_size:
            break
    print(decade, "directory measurements:", rows.get("totalCount"), flush=True)


1950 directory measurements: 1616


1960 directory measurements: 1769


1970 directory measurements: 1473


1980 directory measurements: 1367


1990 directory measurements: 2307


## Fetch and inspect the candidate pages

Parse the overall measurement fields. Preserve missing identity, country, genre, edition, speed and boundary checks as pending. Contributor comments are retained for manual review. The page-stated year is checked against the requested decade but is not treated as an independently verified release year.

Selectors: `header`, `button`, and `:scope > span`. Field labels are more stable than generated CSS classes. Missing or repeated required labels trigger review.

In [5]:
records = []
for candidate in manifest:
    raw = fetch(candidate["url"], candidate["uuid"] + ".html")
    soup = BeautifulSoup(raw, "html.parser", from_encoding="utf-8")
    row = {**candidate, "identity_status": "pending", "measurement_status": "pending",
           "selected_for_main_analysis": False, "raw_run": run_id}
    try:
        headings = [h.get_text(" ", strip=True) for h in soup.select("header")]
        heading = next(h for h in headings if re.search(r"\(\s*\d{4}", h))
        match = re.search(r"^(.*?)\s*\(\s*(\d{4})", heading)
        assert match is not None, "Film heading needs review."
        stats = {}
        for button in soup.select("button"):
            parts = button.select(":scope > span")
            if len(parts) == 2:
                label = parts[0].get_text(strip=True).rstrip(":")
                if label in {"NoS", "LEN", "ASL", "MSL"}:
                    assert label not in stats, "Duplicate overall field."
                    stats[label] = parts[1].get_text(strip=True)
        assert set(stats) == {"NoS", "LEN", "ASL", "MSL"}, "Missing overall field."
        ids = {re.search(r"tt\d+", a["href"]).group() for a in soup.find_all("a", href=re.compile(r"imdb\.com/title/tt\d+"))}
        text = soup.get_text(" ", strip=True)
        comment = re.search(r"comment:\s*(.*?)\s+NoS\s*:", text, re.I)
        row.update(title_detail=match[1].strip(), year_page=int(match[2]),
            shot_count=int(stats["NoS"].replace(",", "")), measured_duration=stats["LEN"],
            asl_seconds=float(stats["ASL"]), msl_seconds=float(stats["MSL"]),
            imdb_ids=sorted(ids), comment=comment[1] if comment else "",
            year_in_requested_decade=candidate["decade"] <= int(match[2]) < candidate["decade"] + 10,
            parse_status="ok")
    except (ValueError, TypeError, AssertionError, StopIteration) as error:
        row.update(parse_status="review", parse_error=type(error).__name__ + ": " + str(error))
    records.append(row)
    print(candidate["decade"], row.get("title_detail", row["title"]), row["parse_status"], flush=True)


1950 Bridges-Go-Round 1 ok


1950 Ugestu ok


1950 Les Achalunes ok


1950 Tic tac ok


1950 Tokyo Story ok


1960 Záhadný pan Hyde ok


... (15 outputs left out)

1990 小武（鏡頭運動+時長） ok


1990 小武（景別+時長） ok


1990 周末情人（鏡頭運動+時長） ok


1990 扁擔·姑娘（景別+時長） ok


## Save the intake audit

These are candidate measurements, not approved full-film samples. The discovery API responses are directory evidence; they do not substitute for the planned Wikidata/Wikipedia metadata and text linkage.

Next, review short works, excerpts, repeat measurements and version notes. Decide the final country/language, genre and sample rules before inspecting trend results. This notebook creates no trend charts or analytical conclusions.

In [6]:
audit = {"run_id": run_id, "window": "1950 <= page year < 2000",
         "selection": "first-page convenience probe; exact title/year/director label dedup only",
         "coverage": coverage, "candidates": records,
         "collector": "AI-assisted acquisition; student review required"}
if collect:
    (run / "intake.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    (folder / "02_audit/decade_intake.json").write_text(json.dumps(audit, ensure_ascii=False, indent=2), encoding="utf-8")
    latest.write_text(run_id, encoding="utf-8")
print("Candidate HTML pages:", len(records))
print("Main-analysis approval: pending for all records")


Candidate HTML pages: 25
Main-analysis approval: pending for all records


## Review film identities

This stage links the 25 saved candidate pages to Wikidata metadata. A shared IMDb ID is the strongest link here. Without it, title, year and director must agree. Search rank alone is not a match.

Identity and measurement scope are separate checks. A matched film can still have an excerpt measurement. No candidate is approved for the main analysis in this stage.

Search variants below are discovery hints. Original source titles stay unchanged. Uncertain corrections stay pending. `review_live = False` replays the saved API responses.

In [7]:
review_live = False
resume_from = ""  # Optional: use a partial run ID to resume live requests.
review_base = folder / "01_raw/identity_review"
review_latest = review_base / "latest_run.txt"
review_id = (resume_from or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")) if review_live else review_latest.read_text().strip()
review_run = review_base / review_id
review_run.mkdir(parents=True, exist_ok=True)
if review_live and not (review_run / "intake_source.json").exists():
    (review_run / "intake_source.json").write_bytes((run / "intake.json").read_bytes())
intake = json.loads((review_run / "intake_source.json").read_bytes())
items = intake["candidates"]

# Reuse only cached bodies whose hashes match successful logged responses.
successful = {}
successful_urls = {}
if (review_run / "requests.jsonl").exists():
    for line in (review_run / "requests.jsonl").read_text().splitlines():
        entry = json.loads(line)
        if entry.get("status") == 200 and entry.get("sha256"):
            file = Path(entry["file"])
            key = entry.get("cache_file") or re.sub(r"_attempt_\d+$", "", file.stem) + file.suffix
            successful[key] = entry["sha256"]
            successful_urls[key] = entry["url"]
reused = set()

def review_bytes(url, name):
    file = review_run / name
    if file.exists() and (not review_live or resume_from):
        body = file.read_bytes()
        if hashlib.sha256(body).hexdigest() == successful.get(name) and successful_urls.get(name) == url:
            reused.add(name)
            return body
        if not review_live:
            raise ValueError("Saved response does not match its logged URL or hash: " + name)
    return fetch(url, name, cache=review_run, online=review_live, interval=5)

api_cache = {}

def api(params, name, host="https://www.wikidata.org/w/api.php"):
    key = (host, json.dumps(params, sort_keys=True))
    if key in api_cache:
        return api_cache[key]
    url = requests.Request("GET", host, params=params).prepare().url
    data = json.loads(review_bytes(url, name))
    if "error" in data:
        raise RuntimeError("API error: " + data["error"].get("code", "unknown"))
    api_cache[key] = data
    return data

print("Identity review:", "resuming cached responses and missing API requests" if review_live and resume_from else "live API requests" if review_live else "saved API responses")
print("Evidence run:", review_id)


Identity review:

 saved API responses
Evidence run: 20261107T173847293157Z


## Find candidate items

The Action API searches labels and aliases. At most five search hits are requested per term. We keep film hits with the page year in their description first, or up to three film hits for review. This bounded search cannot prove that an item is absent from Wikidata. Any search cursor is retained in the raw JSON and review output.

The REST API then supplies the statements used for checking and enrichment. Search descriptions are discovery aids, not identity evidence.

Documentation: [search](https://www.mediawiki.org/wiki/Wikibase/API#wbsearchentities), [REST items](https://www.wikidata.org/wiki/Wikidata:REST_API).

External-ID search uses `haswbstatement:P345=...` as documented in [Wikidata search help](https://www.wikidata.org/wiki/Help:Searching). An empty label search also gets a bounded title/year full-text search. Returned IDs still need REST statement checks.

In [8]:
# These variants only help find candidates. They do not correct the source record.
variants = {
    "886c65b9-54b8-4959-963c-fa71f8367e5e": ("Bridges-Go-Round", "en", "Version suffix omitted for search only."),
    "99c490a5-f3e7-4744-a12c-eb72618e5025": ("Ugetsu", "en", "Possible spelling error; title correction needs review."),
    "7abf9fc0-b3ec-476d-a409-af2e9ff3a037": ("Now!", "en", "Possible punctuation variant."),
    "04b8360e-77e4-4235-9873-1af05ad9b6df": ("Now!", "en", "Director credit omitted for search only."),
    "33de296f-6b14-469c-b0a9-d59ef2e26f9d": ("Mad Max", "en", "Possible spelling error; title correction needs review."),
    "c4ae069f-2fbe-47ea-a88d-6f855cd3e89c": ("Nosferatu the Vampyre", "en", "Possible English title; confirm the source alias."),
    "8acc83dc-883e-41d7-99f0-2bd0043abe20": ("龙猫", "zh", "Measurement annotation removed for search only."),
    "2bfed053-d43b-45d8-9e3b-c5b86419b937": ("小武", "zh", "Measurement annotation removed for search only."),
    "6516d59a-c674-46d6-a826-a2d8a26ee0cf": ("小武", "zh", "Measurement annotation removed for search only."),
    "4e538f6c-e082-41ee-9182-33e201f6d364": ("周末情人", "zh", "Measurement annotation removed for search only."),
    "ad1a1cd0-221e-4517-a352-8e69419ffff4": ("扁担姑娘", "zh", "Simplified title used as a search hint.")
}
searches = {}
choices = {}
for row in items:
    term, language, note = variants.get(row["uuid"], (row["title"].split(" (")[0], "en", "Source title search."))
    key = (term, language)
    if key not in searches:
        name = "search_" + hashlib.sha256((language + term).encode()).hexdigest()[:12] + ".json"
        searches[key] = api({"action": "wbsearchentities", "search": term,
            "language": language, "uselang": "en", "type": "item", "limit": 5,
            "format": "json"}, name)
    result = searches[key]
    hits = result.get("search", [])
    films = [h for h in hits if re.search(r"film|motion picture", h.get("description", ""), re.I)]
    dated = [h for h in films if str(row["year_page"]) in h.get("description", "")]
    picked = dated[:3] if dated else films[:3]
    choices[row["uuid"]] = {"term": term, "language": language, "note": note,
        "search_continue": result.get("search-continue"), "ids": [h["id"] for h in picked]}
    print(row["year_page"], term, "candidate items:", len(picked), flush=True)

# Search by an external ID where available; use a title/year fallback when label search is empty.
for row in items:
    choice = choices[row["uuid"]]
    if row["imdb_ids"]:
        query = " OR ".join("haswbstatement:P345=" + uid for uid in row["imdb_ids"])
    elif not choice["ids"]:
        query = '"' + choice["term"].replace('"', '') + '" "' + str(row["year_page"]) + '"'
    else:
        continue
    name = "lookup_" + hashlib.sha256(query.encode()).hexdigest()[:12] + ".json"
    found = api({"action": "query", "list": "search", "srsearch": query,
        "srnamespace": 0, "srlimit": 5, "format": "json"}, name)
    hits = [h["title"] for h in found.get("query", {}).get("search", [])
            if re.fullmatch(r"Q\d+", h["title"])]
    choice["field_search"] = query
    choice["field_search_continue"] = found.get("continue")
    choice["ids"] = list(dict.fromkeys(choice["ids"] + hits))

entities = {}
for uid in sorted({uid for choice in choices.values() for uid in choice["ids"]}):
    url = "https://www.wikidata.org/w/rest.php/wikibase/v1/entities/items/" + uid
    entity = json.loads(review_bytes(url, uid + ".json"))
    assert entity["id"] == uid
    entities[uid] = entity
print("REST items saved:", len(entities))


1958 Bridges-Go-Round candidate items: 1


1953 Ugetsu candidate items: 1


1959 Les Achalunes candidate items: 0


1957 Tic tac candidate items: 1


1953 Tokyo Story candidate items: 1


1964 Záhadný pan Hyde candidate items: 0


... (16 outputs left out)

1998 小武 candidate items: 1


1995 周末情人 candidate items: 1


1998 扁担姑娘 candidate items: 1


REST items saved: 28


## Read metadata and resolve names

Keep all non-deprecated values, including multiple release years, production countries and runtimes. These can reflect different releases or editions. They are not reduced to a single convenient value.

Related labels make director, country and genre IDs readable. The field mapping is P345 = IMDb, P577 = release dates, P57 = director, P495 = production countries, P136 = genres, P2047 = runtimes. Minute quantities use unit Q7727; other units stay in the raw response.

In [9]:
def values(entity, prop):
    return [s["value"]["content"] for s in entity.get("statements", {}).get(prop, [])
            if s.get("rank") != "deprecated" and s.get("value", {}).get("type") == "value"]

related = sorted({uid for entity in entities.values() for prop in ["P57", "P495", "P136"]
                  for uid in values(entity, prop) if isinstance(uid, str) and re.fullmatch(r"Q\d+", uid)})
labels = {}
for offset in range(0, len(related), 50):
    data = api({"action": "wbgetentities", "ids": "|".join(related[offset:offset + 50]),
        "props": "labels|aliases", "languages": "en|mul|zh|zh-hans|zh-hant|ja|cs|fr|es|de",
        "format": "json"}, f"related_{offset // 50 + 1}.json")
    labels.update(data["entities"])

def name(uid):
    terms = labels.get(uid, {}).get("labels", {})
    return terms.get("en", terms.get("mul", {})).get("value", uid)

metadata = {}
for uid, entity in entities.items():
    dates = values(entity, "P577")
    years = sorted({int(v["time"][1:5]) for v in dates if isinstance(v, dict)
        and v.get("precision", 0) >= 9 and re.match(r"\+\d{4}-", v.get("time", ""))})
    runtimes = [v for v in values(entity, "P2047") if isinstance(v, dict)]
    metadata[uid] = {"wikidata_id": uid, "title_en": entity.get("labels", {}).get("en", ""),
        "description_en": entity.get("descriptions", {}).get("en", ""),
        "imdb_ids_api": values(entity, "P345"), "release_years_api": years,
        "director_ids": values(entity, "P57"),
        "directors_api": [name(q) for q in values(entity, "P57")],
        "country_ids": values(entity, "P495"),
        "countries_api": [name(q) for q in values(entity, "P495")],
        "genre_ids": values(entity, "P136"),
        "genres_api": [name(q) for q in values(entity, "P136")],
        "runtime_quantities_api": runtimes,
        "runtime_minutes_api": sorted({float(v["amount"]) for v in runtimes
            if v.get("unit", "").endswith("/Q7727")}),
        "enwiki": entity.get("sitelinks", {}).get("enwiki", {})}
print("Readable metadata prepared; multiple values retained.")


Readable metadata prepared; multiple values retained.


## Check identity and measurement scope

For a missing shared ID, an exact normalized source-title form, a supported year and a director-name match provide a provisional identity link. Accents, case and punctuation are ignored for this check; source text is preserved. Measurement annotations are removed from title comparison. A possible spelling correction is not silently accepted.

A comment that explicitly says first 30 minutes or names a scene is flagged as an excerpt. Other records stay scope-unverified: lack of a comment does not prove a full-film measurement. Runtime values are displayed as further evidence; no runtime cutoff is chosen here.

In [10]:
import unicodedata
from collections import defaultdict

def normal(text):
    text = unicodedata.normalize("NFKD", text).casefold()
    return "".join(c for c in text if c.isalnum() and not unicodedata.combining(c))

def source_titles(title):
    clean = re.sub(r"[（(][^）)]*(?:景别|景別|鏡頭|镜头|時長|时长)[^）)]*[）)]", "", title)
    parts = [clean, clean.split(" (")[0]]
    parts += re.findall(r"\(([^()]*)\)", clean)
    return {normal(t) for t in parts if t.strip()}

def item_titles(entity):
    terms = list(entity.get("labels", {}).values())
    for group in entity.get("aliases", {}).values():
        terms.extend(group)
    return {normal(t) for t in terms if isinstance(t, str)}

def director_match(source, ids):
    source = normal(str(source))
    for uid in ids:
        person = labels.get(uid, {})
        terms = [v["value"] for v in person.get("labels", {}).values()]
        terms += [v["value"] for group in person.get("aliases", {}).values() for v in group]
        if any(source == normal(t) for t in terms):
            return "exact_name_or_alias"
        if any(len(normal(t)) > 5 and normal(t) in source for t in terms):
            return "name_in_source_credit"
    return "review"

review = []
for source in items:
    checks = []
    for uid in choices[source["uuid"]]["ids"]:
        meta = metadata[uid]
        shared = bool(set(source["imdb_ids"]) & set(meta["imdb_ids_api"]))
        conflict = bool(source["imdb_ids"] and meta["imdb_ids_api"] and not shared)
        title_ok = bool(source_titles(source["title"]) & item_titles(entities[uid]))
        year_ok = source["year_page"] in meta["release_years_api"]
        director = director_match(source.get("director", ""), meta["director_ids"])
        supported = year_ok and not conflict and (shared or title_ok and director != "review")
        checks.append({"wikidata_id": uid, "shared_imdb": shared, "imdb_conflict": conflict,
            "source_title_match": title_ok, "year_supported": year_ok,
            "director_check": director, "identity_supported": supported})
    matches = [check for check in checks if check["identity_supported"]]
    chosen = matches[0] if len(matches) == 1 else None
    status = "pending"
    if chosen:
        status = "shared_imdb_and_year" if chosen["shared_imdb"] else "title_year_director_supported"
    elif len(matches) > 1:
        status = "ambiguous"
    elif any(c["imdb_conflict"] for c in checks):
        status = "imdb_conflict"
    comment = source.get("comment", "")
    excerpt = bool(re.search(r"first\s+30\s+min|only the first 30|\bscene\b", comment, re.I))
    row = {**source, "identity_status": status, "identity_checks": checks,
        "discovery": choices[source["uuid"]], "identity_run": review_id,
        "candidate_metadata": [metadata[uid] for uid in choices[source["uuid"]]["ids"]],
        "measurement_status": "explicit_excerpt" if excerpt else "scope_unverified",
        "measurement_evidence": comment, "selected_for_main_analysis": False}
    if chosen:
        row["metadata"] = metadata[chosen["wikidata_id"]]
    review.append(row)

groups = defaultdict(list)
for row in review:
    if "metadata" in row:
        groups[row["metadata"]["wikidata_id"]].append(row["uuid"])
for row in review:
    uid = row.get("metadata", {}).get("wikidata_id")
    row["same_film_measurement_pages"] = groups.get(uid, [])
    row["duplicate_review"] = "multiple_measurements" if len(groups.get(uid, [])) > 1 else "none_identified"
possible = defaultdict(list)
for row in review:
    key = (normal(row["discovery"]["term"]), row["year_page"])
    possible[key].append(row["uuid"])
for row in review:
    key = (normal(row["discovery"]["term"]), row["year_page"])
    row["possible_same_film_pages"] = possible[key]
    if row["duplicate_review"] == "none_identified" and len(possible[key]) > 1:
        row["duplicate_review"] = "possible_duplicate_not_merged"
print("Identity checks prepared. No measurement selected for analysis.")


Identity checks prepared. No measurement selected for analysis.

## Add English source text

Use the English Wikipedia sitelink for supported identities only. The API returns the article introduction. Its Wikidata page property must agree before text is linked. The full returned introduction is kept, with page ID, revision and attribution URL. This is source text, not a generated synopsis.

Requests are batched in groups of at most 20, following the TextExtracts limit. Continue tokens, if returned, are followed. Missing or short text stays visible for later review. [API documentation](https://www.mediawiki.org/wiki/Extension:TextExtracts#API).

The article request failed after three logged attempts in this run. That failure is replayed as unavailable text; it is not replaced with a generated synopsis.

In [11]:
text_pages = {}
text_failure = review_run / "text_unavailable.json"
titles = sorted({r["metadata"]["enwiki"]["title"] for r in review
    if r.get("metadata", {}).get("enwiki", {}).get("title")})
if text_failure.exists():
    print("English article text unavailable in this run; see the saved failure record.")
for offset in range(0, 0 if text_failure.exists() else len(titles), 20):
    params = {"action": "query", "prop": "extracts|pageprops|info", "inprop": "url",
        "titles": "|".join(titles[offset:offset + 20]), "exintro": 1, "explaintext": 1,
        "exlimit": 20, "redirects": 1, "format": "json", "formatversion": 2}
    part = 1
    while True:
        try:
            data = api(params, f"introductions_{offset // 20 + 1}_{part}.json",
                       "https://en.wikipedia.org/w/api.php")
        except (requests.RequestException, ConnectionError) as error:
            if not review_live:
                raise
            text_failure.write_text(json.dumps({"state": "unavailable",
                "error_type": type(error).__name__, "log": "requests.jsonl"}), encoding="utf-8")
            print("Article request failed; text remains pending. See requests.jsonl.")
            break
        for page in data.get("query", {}).get("pages", []):
            uid = page.get("pageprops", {}).get("wikibase_item")
            if uid:
                text_pages[uid] = {"text_en": page.get("extract", ""),
                    "title": page["title"], "page_id": page["pageid"],
                    "revision_id": page.get("lastrevid"), "url": page.get("fullurl"),
                    "license": "https://creativecommons.org/licenses/by-sa/4.0/"}
        if "continue" not in data:
            break
        params.update(data["continue"])
        part += 1
for row in review:
    uid = row.get("metadata", {}).get("wikidata_id")
    row["source_text"] = text_pages.get(uid, {})
    row["text_status"] = "at_least_50_characters" if len(row["source_text"].get("text_en", "")) >= 50 else "missing_or_short"
print("Article text is linked only when received and its Wikidata ID agrees.")


English article text unavailable in this run; see the saved failure record.
Article text is linked only when received and its Wikidata ID agrees.


## Save and inspect the review

The intake snapshot remains unchanged. This review stores linked metadata, conflicts and pending checks separately. Supported identity is not approval of measurement scope, film format, edition or sample eligibility. Raw country labels are retained alongside API country lists; they are not overwritten.

These first-page candidates are still a convenience probe. The next expansion needs a broader directory frame, documented inclusion rules and film-level deduplication before any decade trend is assessed.

In [12]:
evidence_issues = []
for line in (review_run / "requests.jsonl").read_text().splitlines():
    entry = json.loads(line)
    if not entry.get("sha256"):
        continue
    file = review_run / entry["file"]
    if not file.exists() or hashlib.sha256(file.read_bytes()).hexdigest() != entry["sha256"]:
        evidence_issues.append({"time_utc": entry["time_utc"], "file": entry["file"],
            "status": entry.get("status"), "expected_sha256": entry["sha256"],
            "note": "The early retry helper overwrote this failed response with its successful cache alias. Original log retained; original failed body unavailable. Later attempt files have unique names."})

result = {"review_id": review_id, "source_intake_run": intake["run_id"],
    "stage": "identity and measurement-scope review; no final sample selection",
    "collector": "AI-assisted acquisition and provisional checks; student review required",
    "records": review, "evidence_issues": evidence_issues,
    "text_acquisition": "unavailable_in_this_run" if text_failure.exists() else "received_where_available"}
if review_live:
    body = json.dumps(result, ensure_ascii=False, indent=2)
    (review_run / "review.json").write_text(body, encoding="utf-8")
    (folder / "02_audit/film_review.json").write_text(body, encoding="utf-8")
    review_latest.write_text(review_id, encoding="utf-8")

import pandas as pd
from IPython.display import display
summary = pd.DataFrame([{
    "film": r.get("metadata", {}).get("title_en") or r["title"],
    "year": r["year_page"], "identity": r["identity_status"],
    "measurement": r["measurement_status"], "measured_time": r["measured_duration"],
    "API_runtime_min": r.get("metadata", {}).get("runtime_minutes_api", []),
    "duplicate": r["duplicate_review"], "text": r["text_status"]
} for r in review])
view = summary[["film", "year", "identity", "measurement", "duplicate"]].copy()
view["identity"] = view["identity"].replace({"shared_imdb_and_year": "IMDb + year",
    "title_year_director_supported": "Title + year + director", "pending": "Pending"})
view["measurement"] = view["measurement"].replace({"explicit_excerpt": "Excerpt", "scope_unverified": "Unverified"})
view["duplicate"] = view["duplicate"].replace({"none_identified": "None identified",
    "possible_duplicate_not_merged": "Possible duplicate", "multiple_measurements": "Multiple measurements"})
display(view)
metadata_view = pd.DataFrame([{"film": r["metadata"]["title_en"] or r["title"],
    "HTML_country": r["country"] if isinstance(r["country"], str) else None,
    "API_countries": r["metadata"]["countries_api"], "API_genres": r["metadata"]["genres_api"]}
    for r in review if "metadata" in r])
display(metadata_view.head(3))
# A small view for explaining one scope problem in the video.
for row in review:
    if row["title"] == "Top Gun":
        print("Contributor comment:", row["comment"])
display(summary[summary["film"].eq("Top Gun")][["film", "measured_time", "API_runtime_min", "measurement"]])
print("Main-analysis selection: pending.")

print("Successful response files reused:", len(reused))


,film,year,identity,measurement,duplicate
0,Bridges-Go-Round 1,1958,Pending,Unverified,None identified
1,Ugestu,1953,Pending,Excerpt,None identified
2,Les Achalunes,1959,Pending,Unverified,None identified
3,Tic tac,1957,Pending,Unverified,None identified
4,Tokyo Story,1953,Title + year + director,Unverified,None identified
5,Záhadný pan Hyde,1964,Pending,Unverified,None identified
6,Diego Rivera en la Capilla de Chapingo,1968,Pending,Unverified,None identified
7,The Haunting,1963,Title + year + director,Unverified,None identified
8,Now,1965,Pending,Unverified,Possible duplicate
9,Now - Santiago Alvarez,1965,Pending,Unverified,Possible duplicate


,film,HTML_country,API_countries,API_genres
0,Tokyo Story,Japan,[Japan],[drama film]
1,The Haunting,US,[United Kingdom],"[horror film, psychological horror fiction, LG..."
2,Young Frankenstein,None,[United States],"[comedy horror, horror film, science fiction f..."


Contributor comment: Volleyball scene


,film,measured_time,API_runtime_min,measurement
17,Top Gun,01:16.9,[109.0],explicit_excerpt


Main-analysis selection: pending.
Successful response files reused: 64
